# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
bse = .012167/4

In [3]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')

In [4]:
# other global variables
turn_no_col = 'turn_delta'

In [5]:
FINAL_OUTPUT_LOC = 'individual_corpora'
if not os.path.exists(FINAL_OUTPUT_LOC):
    os.mkdir(FINAL_OUTPUT_LOC)

In [6]:
# to rename the corpus correctly . . .
def lang(x):
    return x.split('-')[1]

#### Load data

In [7]:
# df_ = pd.read_table(FINAL_DOC, sep='\t')
df_ = pd.read_parquet(FINAL_DOC)

In [8]:
df_['self'] = df_['self'].replace({True: 'self', False: 'other'})
df_['self'].loc[df_['null']] = 'null-model'
df_['self'].value_counts()

self
self          8021922
other         6043726
null-model    1267805
Name: count, dtype: int64

In [9]:
for l in df_['lang'].unique():
    sub = df_.loc[df_['lang'].isin([l])]
    print(lang(sub['corpus'].values[0]), sub['conversation_id'].nunique())

cro 164
yid 23
eng 217
fra 60
spa 317
zho 200
deu 120
fin 84
ko 100


In [10]:
df_['corpus'] = [lang(co) for co in tqdm(df_['corpus'])]

  0%|          | 0/15333453 [00:00<?, ?it/s]

In [11]:
df_.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,lang,groups,sample_delta,lang_,resid
0,2.765743,12.0,10.0,2.0,other,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-croation-cro...,1,5,-0.158572
1,2.836436,12.0,7.0,6.0,other,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-croation-cro...,2,5,-0.137692
2,2.906804,12.0,8.0,9.0,other,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-croation-cro...,3,5,-0.050720
3,3.063219,12.0,5.0,10.0,other,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-croation-cro...,4,5,0.055881
4,2.971834,12.0,9.0,12.0,other,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-croation-cro...,5,5,0.030914


In [12]:
df_['corpus'].nunique()

9

## Vis & Other Tests

In [13]:
import plotly.tools as tls
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [14]:
names = ['other', 'self']
main_line_colors = ['royalblue','#ff7f0e']
ribbon_colors = ['rgba(65, 105, 225, 0.2)', 'rgba(255, 127, 14, 0.2)']

In [15]:
df__ = df_.loc[
    (df_[turn_no_col] > 0)
    & (df_[turn_no_col] <= 30)
]

In [16]:
df__ = df__[['self', turn_no_col, 'resid']].groupby(by=['self', turn_no_col]).agg('mean').reset_index()
df__['ste'] = df__[['self', turn_no_col, 'resid']].groupby(by=['self', turn_no_col]).agg('std').reset_index()['resid']
df__['nk'] = df__[['self', turn_no_col, 'resid']].groupby(by=['self', turn_no_col]).agg('count').reset_index()['resid']
df__['SE'] = df__['ste'] / np.sqrt(df__['nk'])

In [17]:
fig2 = go.Figure()

In [18]:
# self
sel = df__['self'] == 'self'
fig2.add_trace(
    go.Scatter(
        y=df__['resid'].loc[sel].values,
        x=df__[turn_no_col].loc[sel].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange',
            # width=4,
            # dash='dash'
        )
    )
)

# upper bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df__[turn_no_col].loc[sel].values,
        y=df__['resid'].loc[sel].values + (2*df__['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='self_upper_bound',
        showlegend=False,
        legendgroup='self',
        fill='tonexty',
        fillcolor='rgba(255, 165, 0, 0.2)',
    ),
    # row=row, col=col
)

# lower bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df__[turn_no_col].loc[sel].values,
        y=df__['resid'].loc[sel].values - (2*df__['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='self_lower_bound',
        showlegend=False,
        legendgroup='self',
        fill='tonexty',
        fillcolor='rgba(255, 165, 0, 0.2)',
    ),
    # row=row, col=col
)

####################################
# other
####################################
sel = df__['self']  == 'other'
fig2.add_trace(
    go.Scatter(
        y=df__['resid'].loc[sel].values,
        x=df__[turn_no_col].loc[sel].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue',
            # width=4,
            # dash='dash'
        )
    )
)

# upper bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df__[turn_no_col].loc[sel].values,
        y=df__['resid'].loc[sel].values + (2*df__['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='other_upper_bound',
        showlegend=False,
        legendgroup='other',
        fill='tonexty',
        fillcolor='rgba(65, 105, 225, 0.2)',
    ),
    # row=row, col=col
)

# lower bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df__[turn_no_col].loc[sel].values,
        y=df__['resid'].loc[sel].values - (2*df__['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='other_lower_bound',
        showlegend=False,
        legendgroup='other',
        fill='tonexty',
        fillcolor='rgba(65, 105, 225, 0.2)',
    ),
    # row=row, col=col
)

####################################
# null
####################################
sel = df__['self'] == 'null-model'
fig2.add_trace(
    go.Scatter(
        y=df__['resid'].loc[sel].values,
        x=df__[turn_no_col].loc[sel].values,
        mode='lines',
        name='null-model',
        showlegend=True,
        legendgroup='null-model',
        line = dict(
            color='#3e474d',
            # width=4,
            # dash='dash'
        )
    )
)

# upper bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df__[turn_no_col].loc[sel].values,
        y=df__['resid'].loc[sel].values + (2*df__['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='null-model_upper_bound',
        showlegend=False,
        legendgroup='null-model',
        fill='tonexty',
        fillcolor='rgba(62, 71, 77, 0.2)',
    ),
    # row=row, col=col
)

# lower bounds for ribbon
fig2.add_trace(
    go.Scatter(
        x=df__[turn_no_col].loc[sel].values,
        y=df__['resid'].loc[sel].values - (2*df__['SE'].loc[sel].values),
        mode='lines',
        line=dict(color='orange',width=0),
        name='null-model_lower_bound',
        showlegend=False,
        legendgroup='null-model',
        fill='tonexty',
        fillcolor='rgba(62, 71, 77, 0.2)',
    ),
    # row=row, col=col
)

In [19]:
fig2.update_layout(template='xgridoff')
fig2.show() 

In [20]:
fig2.write_html(os.path.join(VIS_PATH, 'entropy-delta.html'))

### Individual conversations in a subplot

In [22]:
def timeplot(
        dataframe,
        x_data_col: str,
        y_data_col: str,
        groupby_cols: list = [],
        line_colors: list[tuple] = [],
        line_width: int = 3
):
    df_ = dataframe[groupby_cols + [x_data_col] + [y_data_col]].groupby(by=groupby_cols + [x_data_col]).agg(
        'mean').reset_index()
    df_['std'] = dataframe[groupby_cols + [x_data_col] + [y_data_col]].groupby(by=groupby_cols + [x_data_col]).agg(
        'std').reset_index()[y_data_col]
    df_['K'] = dataframe[groupby_cols + [x_data_col] + [y_data_col]].groupby(by=groupby_cols + [x_data_col]).agg(
        'count').reset_index()[y_data_col]
    df_['se'] = df_['std'] / np.sqrt(df_['K'])
    # print(df_['se'])

    fig = go.Figure()

    for i, condset in enumerate(df_[groupby_cols].drop_duplicates().values):

        condset_name = '-'.join([str(cond) for cond in condset])

        # get where conditions are met
        sel = np.concatenate(
            [
                df_[groupby_cols[j]].isin([v]).values.reshape(-1, 1).astype(int)
                for j, v in enumerate(condset)
            ], axis=-1
        ).sum(axis=-1) == len(condset)

        if (len(line_colors) > 0) and (i < len(line_colors)):
            # get line colors
            line_color = 'rgba({}, {}, {}, 1.0)'.format(*line_colors[i])
            ribbon_color = 'rgba({}, {}, {}, 0.2)'.format(*line_colors[i])
        else:
            line_color = 'rgba(255, 127, 14, 1.0)'
            ribbon_color = 'rgba(255, 127, 14, 0.2)'

        # add mean values plot
        fig.add_trace(
            go.Scatter(
                y=df_[y_data_col].loc[sel].values,
                x=df_[x_data_col].loc[sel].values,
                mode='lines',
                name=condset_name,
                showlegend=True,
                legendgroup=condset_name,
                line=dict(
                    color=line_color,
                    width=line_width
                )
            )
        )

        # add ribbon upper bounds
        fig.add_trace(
            go.Scatter(
                y=df_[y_data_col].loc[sel].values + (2 * df_['se'].loc[sel].values),
                x=df_[x_data_col].loc[sel].values,
                mode='lines',
                name=condset_name + '_upperr_bound',
                showlegend=False,
                legendgroup=condset_name,
                line=dict(
                    color=line_color,
                    width=0
                )
            )
        )

        # add ribbon lower bounds
        fig.add_trace(
            go.Scatter(
                y=df_[y_data_col].loc[sel].values - (2 * df_['se'].loc[sel].values),
                x=df_[x_data_col].loc[sel].values,
                mode='lines',
                name=condset_name + '_lower_bound',
                showlegend=False,
                legendgroup=condset_name,
                line=dict(
                    color=line_color,
                    width=0
                ),
                fill='tonexty',
                fillcolor=ribbon_color
            )
        )

    return fig

#### Actually creating the figure

In [23]:
corpora = [
    'eng', 'deu', 'spa', 'fra', 'zho', 'kor', 
    # 'cro'
]

In [24]:
conversation_id_col = 'corpus'

In [25]:
k_conversations = len(corpora)
k_columns = 2
k_rows = int(np.ceil(k_conversations/k_columns))

In [26]:
assignments = sum([[(row+1,col+1) for col in range(k_columns)] for row in range(k_rows)], [])

### Specific Corpus Visualization

In [27]:
df_['corpus'].value_counts()

corpus
spa    4442726
eng    2256646
cro    2155034
zho    1960903
fra    1340048
kor    1067263
deu     782246
fin      49775
yid      11007
Name: count, dtype: int64

In [28]:
# subplot_fig = make_subplots(
#     rows=k_rows, cols=k_columns,
#     subplot_titles=corpora + [None] * (k_conversations - len(corpora))
#     # row_titles=['Zoom', 'Telephone', 'Instruction', 'BNC/Naturalistic'],
#     # row_titles=['Instruction','Instruction', 'BNC/Naturalistic','BNC/Naturalistic'],
# )

In [29]:
for (i, conversation) in tqdm(enumerate(corpora)):
    df__ = df_.loc[
        df_[conversation_id_col].isin([conversation])
        & (df_[turn_no_col] > 0)
        & (df_[turn_no_col] <= 20)
    ].copy()
    df__['self'] = df__['self'].replace({True: 'self', False: 'other'})
    df__ = df__.sort_values(by=['self'])
    df__.index = range(len(df__))
    
    
    fig = timeplot(
        dataframe=df__,
        x_data_col='turn_delta',
        y_data_col='resid',
        groupby_cols=['self'],
        line_colors=[(65,105,225), (255, 127, 14)]
    ) 
    
    fig.update_layout(
        template='xgridoff',
        margin=dict(
            l=0,
            r=0,
            b=15,
            t=15
        )
    )
    fig.write_html(
        os.path.join(FINAL_OUTPUT_LOC, str(conversation)+'.html')
    )
    
    

0it [00:00, ?it/s]

In [ ]:
# subplot_fig.update_layout(template='xgridoff')
# subplot_fig.show()

In [ ]:
# subplot_fig.write_html(os.path.join(VIS_PATH, 'entropy-delta-individual-languages-3.html'))